# E-commerce Landing Page A/B Test — Analysis

## Pre-registration (written 2026-10-22, before any test results were computed)

**Question:** Does the new landing page change the conversion rate compared with the old page?

| Item | Decision |
|---|---|
| Primary metric | Conversion rate = converted users ÷ users, per group |
| Hypotheses | H0: p_treatment = p_control  ·  H1: p_treatment ≠ p_control |
| Test | Two-proportion z-test, **two-sided**, with a 95% CI for the difference (treatment − control) |
| Significance level | **alpha = 0.05** |
| Main analysis | `ab_dbt.int_ab_users_clean` — 286,690 users; every user appearing more than once in raw is excluded (they may have seen both pages) |
| Sensitivity check | `ab_dbt.int_ab_users_sensitivity` — 290,584 users; mismatched rows dropped, each user's first remaining event kept. Reported alongside the main result, stating whether the conclusion changes |
| Validity check first | Sample ratio mismatch (SRM): chi-square test against a 50/50 split. If p < 0.01, randomization is suspect and the result is not trusted |

**What had been seen before writing this:** row and user counts per group, and the *raw, pre-cleaning*
conversion rates from data profiling on 2026-10-21 (control 12.04%, treatment 11.89%). No conversion rates
or test statistics had been computed on either cleaned table.


## 6a. Pull the clean data

Both user tables come from the dbt models in BigQuery. Before going further, check they match the counts
dbt verified: one row per user, and the expected users per group. Only user counts are shown here;
conversion rates are first computed in 6c.

In [1]:
from google.cloud import bigquery
import pandas as pd

PROJECT = "ab-test-portfolio-512322"
client = bigquery.Client(project=PROJECT)


def load(table):
    sql = f"SELECT * FROM `{PROJECT}.ab_dbt.{table}`"
    return client.query(sql).to_dataframe(create_bqstorage_client=False)


clean = load("int_ab_users_clean")
sens = load("int_ab_users_sensitivity")

# Sanity checks against the dbt-verified counts (users only, no conversion rates yet).
EXPECTED = {
    "clean": {"control": 143_293, "treatment": 143_397},
    "sensitivity": {"control": 145_274, "treatment": 145_310},
}
for name, df in [("clean", clean), ("sensitivity", sens)]:
    users = df["experiment_group"].value_counts().sort_index()
    assert df["user_id"].is_unique, f"{name}: duplicate user_id"
    assert users.to_dict() == EXPECTED[name], f"{name}: {users.to_dict()}"
    print(f"{name:<12} users={len(df):,}  " + "  ".join(f"{g}={n:,}" for g, n in users.items()))

clean        users=286,690  control=143,293  treatment=143,397
sensitivity  users=290,584  control=145,274  treatment=145,310


## 6b. Sample ratio mismatch (SRM) check

Users were meant to be split 50/50. A chi-square test compares the observed group sizes with a perfect
50/50 split. Per the pre-registration, p < 0.01 means randomization is suspect and the result is not trusted.

In [2]:
from scipy.stats import chisquare

SRM_ALPHA = 0.01


def srm_check(name, df):
    obs = df["experiment_group"].value_counts().sort_index()
    stat, p = chisquare(obs.values, [obs.sum() / 2] * 2)
    share = obs["treatment"] / obs.sum()
    verdict = "SRM detected" if p < SRM_ALPHA else "no SRM"
    print(
        f"{name:<12} control={obs['control']:,}  treatment={obs['treatment']:,}  "
        f"diff={obs['treatment'] - obs['control']:+,}  treatment share={share:.3%}  "
        f"chi2={stat:.4f}  p={p:.4f}  -> {verdict}"
    )


srm_check("clean", clean)
srm_check("sensitivity", sens)

clean        control=143,293  treatment=143,397  diff=+104  treatment share=50.018%  chi2=0.0377  p=0.8460  -> no SRM
sensitivity  control=145,274  treatment=145,310  diff=+36  treatment share=50.006%  chi2=0.0045  p=0.9468  -> no SRM


**SRM result: no sample ratio mismatch.** Treatment holds 50.018% of users in the main table
(+104 users, p = 0.846) and 50.006% in the sensitivity table (+36 users, p = 0.947), both far above the
pre-registered 0.01 threshold. A gap this small is what random assignment alone produces, so the split
looks healthy and the comparison in 6c can be trusted on this check. Excluding the 3,894 repeat users
did not unbalance the groups.

## 6c. Two-proportion z-test and 95% CI (pre-registered: two-sided, alpha = 0.05)

Main analysis on `int_ab_users_clean` first, then the sensitivity check on `int_ab_users_sensitivity`.
Lift is treatment − control. The CI is for that difference (Newcombe method, statsmodels default).

In [3]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

ALPHA = 0.05


def ab_test(name, df):
    g = df.groupby("experiment_group")["converted"].agg(conversions="sum", users="count")
    c_conv, c_n = int(g.loc["control", "conversions"]), int(g.loc["control", "users"])
    t_conv, t_n = int(g.loc["treatment", "conversions"]), int(g.loc["treatment", "users"])
    c_rate, t_rate = c_conv / c_n, t_conv / t_n

    z, p = proportions_ztest([t_conv, c_conv], [t_n, c_n], alternative="two-sided")
    low, high = confint_proportions_2indep(
        t_conv, t_n, c_conv, c_n, compare="diff", alpha=ALPHA
    )
    abs_lift = t_rate - c_rate

    print(f"== {name} ==")
    print(f"control:    {c_conv:,} / {c_n:,} = {c_rate:.4%}")
    print(f"treatment:  {t_conv:,} / {t_n:,} = {t_rate:.4%}")
    print(f"absolute lift:  {abs_lift * 100:+.3f} percentage points")
    print(f"relative lift:  {abs_lift / c_rate:+.2%}")
    print(f"z = {z:.3f}, p = {p:.4f}  -> {'significant' if p < ALPHA else 'not significant'} at alpha {ALPHA}")
    print(f"95% CI for difference: [{low * 100:+.3f}, {high * 100:+.3f}] percentage points\n")
    return {
        "control_users": c_n, "treatment_users": t_n,
        "control_rate": c_rate, "treatment_rate": t_rate,
        "abs_lift_pts": abs_lift * 100, "rel_lift": abs_lift / c_rate,
        "z": z, "p": p, "ci_low_pts": low * 100, "ci_high_pts": high * 100,
    }


results = {"clean (main)": ab_test("clean (main)", clean)}
results["sensitivity"] = ab_test("sensitivity", sens)
pd.DataFrame(results).T.round(4)

== clean (main) ==
control:    17,220 / 143,293 = 12.0173%
treatment:  17,025 / 143,397 = 11.8726%
absolute lift:  -0.145 percentage points
relative lift:  -1.20%
z = -1.194, p = 0.2323  -> not significant at alpha 0.05
95% CI for difference: [-0.382, +0.093] percentage points

== sensitivity ==
control:    17,489 / 145,274 = 12.0386%
treatment:  17,264 / 145,310 = 11.8808%
absolute lift:  -0.158 percentage points
relative lift:  -1.31%
z = -1.311, p = 0.1899  -> not significant at alpha 0.05
95% CI for difference: [-0.394, +0.078] percentage points



,control_users,treatment_users,control_rate,treatment_rate,abs_lift_pts,rel_lift,z,p,ci_low_pts,ci_high_pts
clean (main),143293.0,143397.0,0.1202,0.1187,-0.1447,-0.0120,-1.1945,0.2323,-0.3821,0.0927
sensitivity,145274.0,145310.0,0.1204,0.1188,-0.1578,-0.0131,-1.3109,0.1899,-0.3938,0.0781


## 6d. Was the test powered to detect a meaningful effect?

Minimum detectable effect (MDE): the smallest true lift this test had an 80% chance of detecting
(two-sided, alpha = 0.05) with the group sizes we actually have. Then: users per group needed to detect
absolute lifts of +0.5, +1 and +2 points from the control baseline.

In [4]:
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

POWER = 0.80
power_calc = NormalIndPower()


def mde_pts(baseline, n_t, n_c):
    # Solve for Cohen's h, then convert back to an absolute lift above the baseline.
    h = power_calc.solve_power(
        effect_size=None, nobs1=n_t, ratio=n_c / n_t, alpha=ALPHA, power=POWER, alternative="two-sided"
    )
    return (np.sin(np.arcsin(np.sqrt(baseline)) + h / 2) ** 2 - baseline) * 100


for name, r in results.items():
    mde = mde_pts(r["control_rate"], r["treatment_users"], r["control_users"])
    print(
        f"{name:<13} MDE at 80% power: {mde:.3f} points "
        f"({mde / 100 / r['control_rate']:.1%} relative) with ~{r['control_users']:,} users per group"
    )

main = results["clean (main)"]
baseline = main["control_rate"]
print(f"\nUsers per group needed (baseline = control rate {baseline:.4%}):")
for lift in [0.005, 0.01, 0.02]:
    es = proportion_effectsize(baseline + lift, baseline)
    n = power_calc.solve_power(effect_size=es, alpha=ALPHA, power=POWER, ratio=1, alternative="two-sided")
    have = min(main["control_users"], main["treatment_users"])
    print(f"  +{lift * 100:.1f} points: {n:>8,.0f} per group  (we have {have:,}: {have / n:.1f}x)")

clean (main)  MDE at 80% power: 0.342 points (2.8% relative) with ~143,293 users per group
sensitivity   MDE at 80% power: 0.340 points (2.8% relative) with ~145,274 users per group

Users per group needed (baseline = control rate 12.0173%):
  +0.5 points:   67,572 per group  (we have 143,293: 2.1x)
  +1.0 points:   17,183 per group  (we have 143,293: 8.3x)
  +2.0 points:    4,438 per group  (we have 143,293: 32.3x)


**Interpretation (6c + 6d)**

- **Result:** treatment converted at 11.87% vs 12.02% for control, a lift of −0.145 points (−1.2% relative),
  z = −1.19, p = 0.232. Not significant at the pre-registered alpha of 0.05.
- **No effect, or test too small?** The test was **not** too small. With ~143k users per group it had 80%
  power to detect a lift as small as **0.34 points** (2.8% relative), and it had 2–32x the users needed to
  detect +0.5, +1 or +2 points. The 95% CI of **[−0.38, +0.09] points** rules out any improvement larger
  than about +0.09 points. So the data does not prove the effect is exactly zero, but it rules out any
  improvement big enough to matter; the best estimate is slightly negative.
- **Sensitivity check agrees:** keeping the repeat users gives −0.158 points, p = 0.190, CI [−0.39, +0.08].
  Same direction, similar size, same conclusion. Excluding repeat users did not drive the result.